In [0]:
%fs ls dbfs:/FileStore/

path,name,size,modificationTime
dbfs:/FileStore/data_preprocessing_dataset.csv,data_preprocessing_dataset.csv,52243,1753296966000
dbfs:/FileStore/employee_data_10000_with_nulls.csv,employee_data_10000_with_nulls.csv,353265,1753608923000
dbfs:/FileStore/sales_data_10000.csv,sales_data_10000.csv,1208364,1753205623000
dbfs:/FileStore/skewed_sales_data_10000.csv,skewed_sales_data_10000.csv,1200116,1753206713000
dbfs:/FileStore/table/,table/,0,0
dbfs:/FileStore/tables/,tables/,0,0


In [0]:
%pip install great_expectations==0.18.12

Python interpreter will be restarted.
  Attempting uninstall: typing-extensions
    Found existing installation: typing-extensions 4.1.1
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.9/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-130602f5-4f23-4bc4-b0ce-a477b75acb57
    Can't uninstall 'typing-extensions'. No files were found to uninstall.
  Attempting uninstall: attrs
    Found existing installation: attrs 21.4.0
    Not uninstalling attrs at /databricks/python3/lib/python3.9/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-130602f5-4f23-4bc4-b0ce-a477b75acb57
    Can't uninstall 'attrs'. No files were found to uninstall.
  Attempting uninstall: traitlets
    Found existing installation: traitlets 5.1.1
    Not uninstalling traitlets at /databricks/python3/lib/python3.9/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-130602f5-4f23-4bc4-b0ce-a477b75acb57
    Can't uni

In [0]:
# For the Re-validation or the data quality checks let's use null check using great_expectations library:
# Load the great expectations context:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from great_expectations.data_context import get_context
from great_expectations.dataset import SparkDFDataset
context = get_context()

# Create a suite name and add it to context:
suite_name = 'employess_suite_name'
context.add_expectation_suite(
    expectation_suite_name = suite_name
)

# Create a Pyspark DataFrame:
spark = SparkSession.builder.appName('ProgramI').getOrCreate()
df = spark.read.csv('/FileStore/employee_data_10000_with_nulls.csv',header=True,inferSchema=True)

# Detection of missing values in salary column:
df = df.withColumn('salary_d',f.col('Salary').isNull())
missing_values_in_salary = df.filter(df['salary_d'] == True).select(f.count('salary_d')).collect()[0][0]
print(f"Missing values in Salary column : {missing_values_in_salary}")

# Total number of rows:
total_no_of_rows = df.count()
print(f"Total rows : {total_no_of_rows}")

# Percent of missing values in salary column:
missing_value_percent_in_salary = round((missing_values_in_salary/total_no_of_rows)*100,2)
print(f"Missing Value Percent In Salary column : {missing_value_percent_in_salary}")

# Since the missing value % is less than 30%, so applying dropna() to remove missing values:
df = df.dropna(subset=['Salary'],how='any').drop('salary_d')

# Add the expectations to the expectation suite:
df1 = SparkDFDataset(df)
df1.expect_column_values_to_not_be_null(
    column = 'Salary',
    mostly = 0.80
)

Missing values in Salary column : 1696
Total rows : 10200
Missing Value Percent In Salary column : 16.63
Out[1]: {
  "success": true,
  "expectation_config": {
    "expectation_type": "expect_column_values_to_not_be_null",
    "kwargs": {
      "column": "Salary",
      "mostly": 0.8,
      "result_format": "BASIC"
    },
    "meta": {}
  },
  "result": {
    "element_count": 8504,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "partial_unexpected_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [0]:
# Save the expectation suite to given directory:
context.save_expectation_suite(
    expectation_suite = df1.get_expectation_suite(),
    expectation_suite_name = suite_name
)

In [0]:
"""
Since we have stored the expectation suite into the .gx hidden temp directory instead of manually creating a directory and then configuring it with DataContext and then saving the expectation suite to that directory. 
"""
# Create a datasource for the RuntimeBatchRequest:
context.add_or_update_datasource(
    name="my_pyspark_datasource",
    class_name="Datasource",
    execution_engine={
        "class_name": "SparkDFExecutionEngine"
    },
    data_connectors={
        "runtime_data_connector": {
            "class_name": "RuntimeDataConnector",
            "batch_identifiers": ["default_identifier_name"]
        }
    }
)

Out[3]: <great_expectations.datasource.new_datasource.Datasource at 0x7f41b343b250>

In [0]:
# List the datasources for the RuntimeBatchRequest:
datasources = context.list_datasources()
print(datasources)

[{'name': 'my_pyspark_datasource', 'class_name': 'Datasource', 'module_name': 'great_expectations.datasource', 'execution_engine': {'class_name': 'SparkDFExecutionEngine', 'module_name': 'great_expectations.execution_engine'}, 'data_connectors': {'runtime_data_connector': {'class_name': 'RuntimeDataConnector', 'module_name': 'great_expectations.datasource.data_connector', 'batch_identifiers': ['default_identifier_name']}}}]


In [0]:
# Create a RuntimeBatchRequest to conifgure the great expectations with the Pyspark DataFrame to validate:
from great_expectations.core.batch import RuntimeBatchRequest
batchrequest = RuntimeBatchRequest(
    datasource_name="my_pyspark_datasource",
    data_connector_name="runtime_data_connector",
    data_asset_name="pyspark_df_asset",  # Arbitrary name
    runtime_parameters={"batch_data": df},  # Your existing DataFrame
    batch_identifiers={"default_identifier_name": "default_id"}
)

In [0]:
# Now apply the expectation suite to the Pyspark DataFrame using validator:
validator = context.get_validator(
    batch_request=batchrequest,
    expectation_suite_name=suite_name
)

In [0]:
# Now apply the validate() on the validator:
results = validator.validate()
print(results)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "results": [
    {
      "success": true,
      "expectation_config": {
        "expectation_type": "expect_column_values_to_not_be_null",
        "kwargs": {
          "column": "Salary",
          "mostly": 0.8,
          "batch_id": "eeb0f3126901bc85361f8c9bb0156a7e"
        },
        "meta": {}
      },
      "result": {
        "element_count": 8504,
        "unexpected_count": 0,
        "unexpected_percent": 0.0,
        "partial_unexpected_list": []
      },
      "meta": {},
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message": null
      }
    }
  ],
  "evaluation_parameters": {},
  "statistics": {
    "evaluated_expectations": 1,
    "successful_expectations": 1,
    "unsuccessful_expectations": 0,
    "success_percent": 100.0
  },
  "meta": {
    "great_expectations_version": "0.18.12",
    "expectation_suite_name": "employess_suite_name",
    "run_id": {
      "run_name": null,

In [0]:
list_1 = results['results']
print(list_1)

[{
  "success": true,
  "expectation_config": {
    "expectation_type": "expect_column_values_to_not_be_null",
    "kwargs": {
      "column": "Salary",
      "mostly": 0.8,
      "batch_id": "eeb0f3126901bc85361f8c9bb0156a7e"
    },
    "meta": {}
  },
  "result": {
    "element_count": 8504,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}]


In [0]:
# Perform Data Mining and get the extracted JSON out:
list_1 = results['results']
my_list, my_dict = [],list_1[0]
my_list.append({
    'success': my_dict['success'],
    'expectation_type': my_dict['expectation_config']['expectation_type'],
    'column_name': my_dict['expectation_config']['kwargs']['column'],
    'mostly': my_dict['expectation_config']['kwargs']['mostly'],
    'batch_identifier': my_dict['expectation_config']['kwargs']['batch_id'],
    'element_count': my_dict['result']['element_count'],
    'unexpected_count': my_dict['result']['unexpected_count'],
    'unexpected_percent': my_dict['result']['unexpected_percent'],
    'raised_exception': my_dict['exception_info']['raised_exception']
    })
print(f"Json Parsed : \n{my_list}")

Json Parsed : 
[{'success': True, 'expectation_type': 'expect_column_values_to_not_be_null', 'column_name': 'Salary', 'mostly': 0.8, 'batch_identifier': 'eeb0f3126901bc85361f8c9bb0156a7e', 'element_count': 8504, 'unexpected_count': 0, 'unexpected_percent': 0.0, 'raised_exception': False}]


In [0]:
# Convert the parsed json into Pyspark DataFrame:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Program_I').getOrCreate()
df = spark.createDataFrame(my_list)
df.display()

batch_identifier,column_name,element_count,expectation_type,mostly,raised_exception,success,unexpected_count,unexpected_percent
eeb0f3126901bc85361f8c9bb0156a7e,Salary,8504,expect_column_values_to_not_be_null,0.8,false,true,0,0.0


In [0]:
# Export the new Pyspark DataFrame into Path based delta table which is un-registered:
df.write.format('delta').save('/user/hive/warehouse/validatedData')

In [0]:
# Register the un-register delta table:
table_name = "validatedData"
table_path = "/user/hive/warehouse/validatedData"

spark.sql(f"""
    CREATE TABLE {table_name}
    USING DELTA
    LOCATION '{table_path}'
""")

Out[39]: DataFrame[]

In [0]:
df1 = spark.read.table('default.validatedData')
df1.display()

batch_identifier,column_name,element_count,expectation_type,mostly,raised_exception,success,unexpected_count,unexpected_percent
eeb0f3126901bc85361f8c9bb0156a7e,Salary,8504,expect_column_values_to_not_be_null,0.8,false,true,0,0.0


In [0]:
# Export the data from managed delta table to a parquet file in any of the folder:
df1.write.mode('overwrite').parquet('/FileStore/table/validateddata')